YOLOv8m + ECA+CBAM

Environment

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if ON_KAGGLE:
    ROOT = "/kaggle/working"
elif ON_COLAB:
    ROOT = "/content"
else:
    ROOT = "."
print("Running on:", "KAGGLE" if ON_KAGGLE else "COLAB" if ON_COLAB else "LOCAL")
OUTPUT_DIR = os.path.join(ROOT, "attention_results")
SAVE_DIR = os.path.join(ROOT, "saved_models")
for d in [OUTPUT_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Output : {OUTPUT_DIR}")
print(f"Models : {SAVE_DIR}")


Running on: LOCAL
Output : ./attention_results
Models : ./saved_models


In [2]:
# Installs kagglehub, matplotlib, pillow and numpy.
!pip install kagglehub matplotlib pillow numpy -q

Verify Environment

In [3]:
# Prints package versions and confirms GPU availability, device name and VRAM.
import torch, numpy as np, pandas as pd, cv2, random, gc

print(f"PyTorch : {torch.__version__}")
print(f"OpenCV  : {cv2.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print("All packages OK")
except ImportError as e:
    print(f"Missing: {e}")
    print("Run: pip install ultralytics seaborn tqdm kagglehub pyyaml")


PyTorch : 2.10.0+cu128
OpenCV  : 4.13.0
Device  : cuda
GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.4 GB
All packages OK


Config

In [4]:
# Defines the training and evaluation config: two-phase epochs and learning rates, batch size, image size, weight decay, worker count, seeds, CBAM hyperparameters, and the confidence sweep grid.
EPOCHS_FROZEN = 10
EPOCHS_FULL = 40
BATCH = 16
IMG_SIZE = 640
LR_FROZEN = 1e-3
LR_FULL = 2e-4
WEIGHT_DECAY = 5e-4


NUM_WORKERS = 2

# Multi-seed
SEEDS = [42, 123, 456]

# Attention
CBAM_REDUCTION = 16
CBAM_KERNEL = 7

# Evaluation
CONF_SWEEP = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]
CONF_DEFAULT = 0.25
IOU_THRESH = 0.5

 Dataset + MD5 Deduplication

In [5]:
# Downloads the three Kaggle datasets, loads every annotation through the unified loader, and removes duplicates by MD5 hash before any split.
import kagglehub, shutil, yaml
from pathlib import Path
import xml.etree.ElementTree as ET

print("Downloading datasets ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


def load_annotated_potholes(root):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        [
                            float(bb.find("xmin").text),
                            float(bb.find("ymin").text),
                            float(bb.find("xmax").text),
                            float(bb.find("ymax").text),
                        ]
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


import pandas as pd


def load_ashishkumar_csv(root):
    root = Path(root)
    df = pd.read_csv(root / "train" / "labels.csv")
    grouped = df.groupby("ImageID")
    records = []
    for img_path in sorted((root / "train" / "images").glob("*.jpg")):
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    [
                        float(row["XMin"]),
                        float(row["YMin"]),
                        float(row["XMax"]),
                        float(row["YMax"]),
                    ]
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    recs = (
        load_ashishkumar_csv(root)
        if name == "ashishkumar"
        else load_annotated_potholes(root)
    )
    print(
        f"  {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

print(f"Before dedup: {len(all_records)}")


import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in all_records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

records = [r for r, keep in zip(annotated_only, keep_mask) if keep]
print(f"After dedup: {len(records)}")
print(f"Duplicates removed: {len(all_records) - len(records)}")


Datasets ready
  chitholian: 665 images (1740 gt boxes)
  andrewmvd: 665 images (1740 gt boxes)
  ashishkumar: 674 images (1371 gt boxes)
Before dedup: 2004
Computing normalized pixel arrays for dedup ...
After dedup: 926
Duplicates removed: 1078


Build YOLO Dataset (fixed split, seed=42)

In [6]:
# Shuffles the deduplicated records under seed 42, splits 80/20, and writes the YOLO-format image and label directories plus data.yaml.
random.seed(42)
np.random.seed(42)
random.shuffle(records)
split_idx = int(len(records) * 0.8)
train_recs, val_recs = records[:split_idx], records[split_idx:]
print(f"Train: {len(train_recs)}  Val: {len(val_recs)}")

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
DATA_YAML = f"{YOLO_DIR}/data.yaml"
for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


def convert_to_yolo(rec_list, split):
    written = 0
    for rec in rec_list:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            continue
        h, w = img.shape[:2]
        dst = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        if not os.path.exists(dst):
            shutil.copy(str(rec["image_path"]), dst)
        with open(f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt", "w") as f:
            for box in rec["gt_boxes"]:
                x1, y1, x2, y2 = box
                f.write(
                    f"0 {((x1 + x2) / 2) / w:.6f} {((y1 + y2) / 2) / h:.6f} "
                    f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}\n"
                )
        written += 1
    return written


n_tr = convert_to_yolo(train_recs, "train")
n_va = convert_to_yolo(val_recs, "val")
with open(DATA_YAML, "w") as f:
    yaml.dump(
        {
            "path": YOLO_DIR,
            "train": "images/train",
            "val": "images/val",
            "nc": 1,
            "names": ["pothole"],
        },
        f,
    )
print(f"YOLO dataset: train={n_tr}  val={n_va}")


Train: 740  Val: 186
YOLO dataset: train=740  val=186
